In [1]:
# --- Libraries
import os
import logging
import random
import warnings
import gc

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision
from sklearn.model_selection import KFold 

2026-03-21 22:19:54.964474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774131595.189861      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774131595.252085      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# --- Configuration 

INPUT_ROOT = '/kaggle/input/csiro-biomass'
TRAIN_CSV  = os.path.join(INPUT_ROOT, 'train.csv')
TEST_CSV   = os.path.join(INPUT_ROOT, 'test.csv')

IMG_SIZE   = 256

BATCH_SIZE_PER_REPLICA = 16

# 2 Gpus
STRATEGY = tf.distribute.MirroredStrategy()
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * STRATEGY.num_replicas_in_sync

print(STRATEGY.num_replicas_in_sync)
print(GLOBAL_BATCH_SIZE)

# Mixed precision
mixed_precision.set_global_policy('mixed_float16')
print(mixed_precision.global_policy())

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2
32
<DTypePolicy "mixed_float16">


I0000 00:00:1774131612.682458      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774131612.685059      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
# --- Data prep
raw_train_df = pd.read_csv(TRAIN_CSV)

# Pivot long to wide
train_df = raw_train_df.pivot(index='image_path', columns='target_name', values='target').reset_index()

# Paths (add the complete position of each image)
train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(INPUT_ROOT, x))

In [4]:
raw_train_df

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750
...,...,...,...,...,...,...,...,...,...
1780,ID983582017__Dry_Clover_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Clover_g,0.0000
1781,ID983582017__Dry_Dead_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Dead_g,0.0000
1782,ID983582017__Dry_Green_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Green_g,40.9400
1783,ID983582017__Dry_Total_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Total_g,40.9400


In [5]:
train_df

target_name,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,full_path
0,train/ID1011485656.jpg,0.0000,31.9984,16.2751,48.2735,16.2750,/kaggle/input/csiro-biomass/train/ID1011485656...
1,train/ID1012260530.jpg,0.0000,0.0000,7.6000,7.6000,7.6000,/kaggle/input/csiro-biomass/train/ID1012260530...
2,train/ID1025234388.jpg,6.0500,0.0000,0.0000,6.0500,6.0500,/kaggle/input/csiro-biomass/train/ID1025234388...
3,train/ID1028611175.jpg,0.0000,30.9703,24.2376,55.2079,24.2376,/kaggle/input/csiro-biomass/train/ID1028611175...
4,train/ID1035947949.jpg,0.4343,23.2239,10.5261,34.1844,10.9605,/kaggle/input/csiro-biomass/train/ID1035947949...
...,...,...,...,...,...,...,...
352,train/ID975115267.jpg,40.0300,0.0000,0.8000,40.8300,40.8300,/kaggle/input/csiro-biomass/train/ID975115267.jpg
353,train/ID978026131.jpg,24.6445,4.1948,12.0601,40.8994,36.7046,/kaggle/input/csiro-biomass/train/ID978026131.jpg
354,train/ID980538882.jpg,0.0000,1.1457,91.6543,92.8000,91.6543,/kaggle/input/csiro-biomass/train/ID980538882.jpg
355,train/ID980878870.jpg,32.3575,0.0000,2.0325,34.3900,34.3900,/kaggle/input/csiro-biomass/train/ID980878870.jpg


In [6]:
# Scale Targets

# Scale Targets 
target_cols = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']   # Only these 3 bcs they should be the easist to predic (the others with math)
MAX_VAL = train_df[target_cols].max().max()             # Max value for scaling
TARGET_SCALER = MAX_VAL * 1.1

# Arrays
X_paths = train_df['full_path'].values                  
Y_targets = train_df[target_cols].values.astype('float32') / TARGET_SCALER   # Scaling

In [7]:
print(X_paths[:5])
print("\n")
print(Y_targets[:5])

['/kaggle/input/csiro-biomass/train/ID1011485656.jpg'
 '/kaggle/input/csiro-biomass/train/ID1012260530.jpg'
 '/kaggle/input/csiro-biomass/train/ID1025234388.jpg'
 '/kaggle/input/csiro-biomass/train/ID1028611175.jpg'
 '/kaggle/input/csiro-biomass/train/ID1035947949.jpg']


[[0.23632202 0.07967396 0.07967445]
 [0.03720566 0.03720566 0.03720566]
 [0.02961766 0.02961766 0.        ]
 [0.27026924 0.11865472 0.11865472]
 [0.16734909 0.05365692 0.05153033]]


In [8]:
# --- Load & pre-process

@tf.function
def load_image(path, label):
    # Load
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels = 3)
    # Resize
    img = tf.image.resize(img,
                          [IMG_SIZE, IMG_SIZE],
                          preserve_aspect_ratio = False,
                          antialias = True
                         )
    # cast
    img = tf.cast(img, tf.float32)
    return img, label

@tf.function
def augment_geometric(img, label):
    # Random Flips
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)

    # Random Rotate
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    img = tf.image.rot90(img, k)

    return img, label

# MixUp
MIXUP_LAYER = keras.layers.MixUp(alpha=0.034272, dtype='float32')

@tf.function
def apply_mixup(images, labels):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    outputs = MIXUP_LAYER({"images": images, "labels": labels})
    return outputs["images"], outputs["labels"]

# Model 
* Backbone: ConvNeXt-Base with Squeeze-and-Excitation attention
* Pooling: Learnable GeM Pooling (initial p=3.0)
* Head: Multi-branch regression structure with three dedicated paths for Total, GDM and Green biomass.
* Regularization: Combined Gaussian Noise, Dropout (0.3) and Swish activations

The model and the parameters were obtained with an Optuna Search.

In [9]:
# Generalized mean pooling (GeM)

@tf.keras.utils.register_keras_serializable()
class GeMPooling2D(layers.Layer):
    def __init__(self, p_init=2.24828, **kwargs):
        super().__init__(**kwargs)
        self.p_init = p_init

    def build(self, input_shape):
        self.p = self.add_weight(
            name="p_param",
            shape=(1,),
            initializer=keras.initializers.Constant(self.p_init),
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        # (1/N * sum(x^p))^(1/p)
        x = tf.maximum(inputs, 1e-6)
        x = tf.pow(x, self.p)
        x = tf.reduce_mean(x, axis=[1, 2], keepdims=False)
        return tf.pow(x, 1.0 / self.p)

    # save/load the p_init value
    def get_config(self):
        config = super().get_config()
        config.update({"p_init": self.p_init})
        return config

In [10]:
# Attention block

def attention_block(inputs, ratio=16):
    """
    Squeeze-and-Excitation Block
    """
    # Number of features
    channels = inputs.shape[-1]
    # Squeeze: global Average Pooling to get a summary vector (Squeeze feature from (H, W, C) to (1, 1, C))
    x = layers.GlobalAveragePooling2D(keepdims=True)(inputs)
    # Excitation: bottleneck part, reduce param to learn (weights)
    x = layers.Dense(channels // ratio, activation='swish')(x)
    # Restoration: restore to the original dimension and weights can be put into the original feature map
    x = layers.Dense(channels, activation='sigmoid')(x)
    # Scale: multiply the original inputs by the calculated weights
    return layers.Multiply()([inputs, x])

In [11]:
# --- Model (+ Data Aug)

weights_path = '/kaggle/input/models/marcorosato/convnext-small/tensorflow2/default/1/convnext_small_notop.h5'

def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Backbone
    base = keras.applications.ConvNeXtSmall(
        include_top=False,
        weights=None,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        name = "convnext_backbone"
    )

    base.load_weights(weights_path)
    base.trainable = False

    x = base(inputs)
    x = attention_block(x)

    x = GeMPooling2D(p_init=2.24828)(x)

    # Gaussian Noise to regularize
    x = layers.GaussianNoise(0.037071)(x)

    # Dense layer + Dropout
    x = layers.Dense(384, activation='swish')(x)
    x = layers.Dropout(0.39424)(x)
    
    # Common dense layer 
    shared = layers.Dense(128, activation='swish')(x)

    # # Multi-Head Outputs
    # Head 1: Dry_Total_g
    h1 = layers.Dense(128, activation='swish')(shared)
    drop_h1 = layers.Dropout(0.29078)(h1)
    out1 = layers.Dense(1, activation='linear', name='total_head', dtype='float32')(drop_h1)
    # Head 2: GDM_g
    h2 = layers.Dense(128, activation='swish')(shared)
    drop_h2 = layers.Dropout(0.29078)(h2)
    out2 = layers.Dense(1, activation='linear', name='gdm_head', dtype='float32')(drop_h2)
    # Head 3: Dry_Green_g
    h3 = layers.Dense(128, activation='swish')(shared)
    drop_h3 = layers.Dropout(0.29078)(h3)
    out3 = layers.Dense(1, activation='linear', name='green_head', dtype='float32')(drop_h3)

    # Concatenate
    outputs = layers.Concatenate(axis=-1)([out1, out2, out3])

    return keras.Model(inputs, outputs)

model = build_model()

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_backbone   │ (None, 8, 8, 768) │ 49,454,688 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1, 1, 768) │          0 │ convnext_backbon… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1, 1, 48)  │     36,912 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1, 1, 768) │     37,632 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 8, 8, 768) │          0 │ convnext_backbon… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ge_m_pooling2d      │ (None, 768)       │          1 │ multiply[0][0]    │
│ (GeMPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise      │ (None, 768)       │          0 │ ge_m_pooling2d[0… │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 384)       │    295,296 │ gaussian_noise[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 384)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     49,280 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     16,512 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     16,512 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     16,512 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_1 (Cast)       │ (None, 128)       │          0 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_2 (Cast)       │ (None, 128)       │          0 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_3 (Cast)       │ (None, 128)       │          0 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ total_head (Dense)  │ (None, 1)         │        129 │ cast_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gdm_head (Dense)    │ (None, 1)         │        129 │ cast_2[0][0]    

 Total params: 49,923,732 (190.44 MB)

 Trainable params: 469,044 (1.79 MB)

 Non-trainable params: 49,454,688 (188.65 MB)

In [12]:
# --- Test data

raw_test_df = pd.read_csv(TEST_CSV)
unique_test_df = raw_test_df.drop_duplicates(subset=['image_path'])
test_paths = [os.path.join(INPUT_ROOT, p) for p in unique_test_df['image_path'].values]

@tf.function
def load_test_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img

test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(load_test_image, num_parallel_calls=tf.data.AUTOTUNE).batch(GLOBAL_BATCH_SIZE)


# Test-Time Augmentation (TTA)
def predict_with_tta(model, dataset):
    """
    Data aug on test to improve results by average the results
    """
    p1 = model.predict(dataset, verbose=0)
    p2 = model.predict(dataset.map(lambda x: tf.image.flip_left_right(x)), verbose=0)  
    p3 = model.predict(dataset.map(lambda x: tf.image.flip_up_down(x)), verbose=0)
    return (p1 + p2 + p3) / 3.0

In [13]:
# --- K-fold Training Loop

N_FOLDS = 5

kf = KFold(n_splits = N_FOLDS, shuffle = True, random_state = 42)

fold_predictions = [] # Store predictions each fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X_paths, Y_targets)):
    print(f"\n{'='*15} FOLD {fold+1}/{N_FOLDS} {'='*15}")
    
    # Split Data
    X_train, Y_train = X_paths[train_idx], Y_targets[train_idx]
    X_val, Y_val     = X_paths[val_idx],   Y_targets[val_idx]

    # Train dataset
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)         
    train_ds = train_ds.cache()                                                      
    train_ds = train_ds.shuffle(len(X_train), seed=42)                               
    train_ds = train_ds.map(augment_geometric, num_parallel_calls=tf.data.AUTOTUNE)  
    train_ds = train_ds.batch(GLOBAL_BATCH_SIZE)                                     
    train_ds = train_ds.map(apply_mixup, num_parallel_calls=tf.data.AUTOTUNE)        
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

    # Val dataset
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.cache()
    val_ds = val_ds.batch(GLOBAL_BATCH_SIZE)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

    
    keras.backend.clear_session()

    # Train Head

    print(f"\n--- Fold {fold+1}: Training Head ---")
    
    with STRATEGY.scope():
        
        model = build_model()
        
        optimizer_1 = tf.keras.optimizers.AdamW(
        learning_rate = 7e-4, 
        weight_decay = 0.0037214
        )
       
        model.compile(optimizer = optimizer_1,
                      loss=keras.losses.Huber(delta=0.84009),
                      metrics = ['mae']
                     )
    
        early_stop = keras.callbacks.EarlyStopping(monitor = 'val_loss',
                                           patience = 7,
                                           restore_best_weights = True
                                          )
        
        reduce_plateau = keras.callbacks.ReduceLROnPlateau(monitor = "val_loss",
                                                           factor = 0.1,
                                                           patience = 2,
                                                           verbose = 0                                                        
        )
        
    
    model.fit(train_ds,
              validation_data = val_ds,
              epochs = 20,
              callbacks = [early_stop, reduce_plateau],
              verbose = 1
             )


    
    # Fine Tune

    print(f"\n--- Fold {fold+1}: Fine Tuning ---")
    
    model.get_layer("convnext_backbone").trainable = True
    
    FINE_TUNE_BATCH_SIZE = GLOBAL_BATCH_SIZE // 2

    # Train
    ft_train_ds = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    ft_train_ds = ft_train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).cache()
    ft_train_ds = ft_train_ds.shuffle(len(X_train), seed=42)
    ft_train_ds = ft_train_ds.map(augment_geometric, num_parallel_calls=tf.data.AUTOTUNE)
    ft_train_ds = ft_train_ds.batch(FINE_TUNE_BATCH_SIZE) 
    ft_train_ds = ft_train_ds.map(apply_mixup, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    # Val
    ft_val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
    ft_val_ds = ft_val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).cache()
    ft_val_ds = ft_val_ds.batch(FINE_TUNE_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
            
    with STRATEGY.scope():

        # Cosine schedule -> Optuna LR & Cosine Decay
        lr_schedule_2 = keras.optimizers.schedules.CosineDecay(
            initial_learning_rate = 0.00006841,
            decay_steps = 1498,
            alpha = 0.01
        )
        
        optimizer_2 = tf.keras.optimizers.AdamW(
            learning_rate = lr_schedule_2, 
            weight_decay = 0.0037214
        )
    
        
        model.compile(optimizer = optimizer_2,
                      loss=keras.losses.Huber(delta=0.84009),
                      metrics = ['mae']
                     )
    
    ckpt_path = f"model_fold_{fold}.keras"
    ckpt = keras.callbacks.ModelCheckpoint(ckpt_path,              # Save Best Model Checkpoint
                                           save_best_only = True,
                                           monitor = 'val_loss',
                                           verbose = 0
                                          )  
    
    history = model.fit(ft_train_ds,
                        validation_data = ft_val_ds,
                        epochs = 30,
                        callbacks = [ckpt, early_stop],
                        verbose = 1
                       )


    # Memory clean up
    del model
    keras.backend.clear_session()
    gc.collect()
    
    
    # Predict on test data
    print(f"Predicting Fold {fold+1}")
    with STRATEGY.scope():
        
        best_model = keras.models.load_model(
            ckpt_path,          # Load best weights
            custom_objects={"GeMPooling2D": GeMPooling2D}
        ) 
        
    preds = predict_with_tta(best_model, test_ds)
    fold_predictions.append(preds)
    
    # Cleanup
    del best_model, train_ds, val_ds
    import gc; gc.collect()


=============== FOLD 1/5 ===============

--- Fold 1: Training Head ---
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
Epoch 1/20
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 21 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs

I0000 00:00:1774131653.826725      62 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1774131654.688433      63 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1774131655.110621      62 service.cc:148] XLA service 0x7d3d296d97d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774131655.111583      62 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774131655.111602      62 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1774131655.541290      62 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
E0000 00:00:1774131658.788065      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1774131658.802422      63 gpu_timer.cc:82] Delay kernel timed out: me

8/9 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - loss: 0.0279 - mae: 0.1933

E0000 00:00:1774131667.683774      61 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1774131667.687819      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1774131667.825723      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1774131667.827356      61 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1774131667.963115      62 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:0

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 905ms/step - loss: 0.0280 - mae: 0.1932INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


...

W0000 00:00:1774133851.739050      20 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Conv2D" attr { key: "T" value { type: DT_FLOAT } } attr { key: "data_format" value { s: "NCHW" } } attr { key: "dilations" value { list { i: 1 i: 1 i: 1 i: 1 } } } attr { key: "explicit_paddings" value { list { } } } attr { key: "padding" value { s: "VALID" } } attr { key: "strides" value { list { i: 1 i: 1 i: 4 i: 4 } } } attr { key: "use_cudnn_on_gpu" value { b: true } } inputs { dtype: DT_FLOAT shape { dim { } dim { size: 3 } dim { size: 256 } dim { size: 256 } } } inputs { dtype: DT_FLOAT shape { dim { size: 4 } dim { size: 4 } dim { size: 3 } dim { size: 96 } } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "12050" } environment { key: "cudnn" value: "90300" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multi


=============== FOLD 4/5 ===============

--- Fold 4: Training Head ---
Epoch 1/20
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 21 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 739ms/step - loss: 0.0276 - mae: 0.1787INFO:tensorflow:Collective all_reduce tensors: 1

W0000 00:00:1774134768.637288      20 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Conv2D" attr { key: "T" value { type: DT_FLOAT } } attr { key: "data_format" value { s: "NCHW" } } attr { key: "dilations" value { list { i: 1 i: 1 i: 1 i: 1 } } } attr { key: "explicit_paddings" value { list { } } } attr { key: "padding" value { s: "VALID" } } attr { key: "strides" value { list { i: 1 i: 1 i: 4 i: 4 } } } attr { key: "use_cudnn_on_gpu" value { b: true } } inputs { dtype: DT_FLOAT shape { dim { } dim { size: 3 } dim { size: 256 } dim { size: 256 } } } inputs { dtype: DT_FLOAT shape { dim { size: 4 } dim { size: 4 } dim { size: 3 } dim { size: 96 } } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "12050" } environment { key: "cudnn" value: "90300" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multi


=============== FOLD 5/5 ===============

--- Fold 5: Training Head ---
Epoch 1/20
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 21 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 1 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 714ms/step - loss: 0.0206 - mae: 0.1659INFO:tensorflow:Collective all_reduce tensors: 1

W0000 00:00:1774135444.853710      20 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Conv2D" attr { key: "T" value { type: DT_FLOAT } } attr { key: "data_format" value { s: "NCHW" } } attr { key: "dilations" value { list { i: 1 i: 1 i: 1 i: 1 } } } attr { key: "explicit_paddings" value { list { } } } attr { key: "padding" value { s: "VALID" } } attr { key: "strides" value { list { i: 1 i: 1 i: 4 i: 4 } } } attr { key: "use_cudnn_on_gpu" value { b: true } } inputs { dtype: DT_FLOAT shape { dim { } dim { size: 3 } dim { size: 256 } dim { size: 256 } } } inputs { dtype: DT_FLOAT shape { dim { size: 4 } dim { size: 4 } dim { size: 3 } dim { size: 96 } } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "12050" } environment { key: "cudnn" value: "90300" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multi

In [14]:
# --- Ensamble avereging

# Average the predictions
avg_preds_scaled = np.mean(fold_predictions, axis=0)

# Clip & Convert to Grams (denormalize)
preds_3_grams = np.clip(avg_preds_scaled, 0, None) * TARGET_SCALER

In [15]:
# --- Physics & Submission 
final_preds_5 = []        # there are 5 things to predict

for row in preds_3_grams:
    total = row[0]        # Dry_Total_g  (total biomass)
    gdm   = row[1]        # GDM_g        (green dry matter)
    green = row[2]        # Dry_Green_g  (dry green biomass)
    
    # Physics Constraints

    if gdm > total:       # gdm must be <= than total biomass
        gdm = total       

    if green > gdm:       # dry green must be <= than gdm 
        green = gdm
        
    dead = total - gdm              # dry dead = total biomass - green dry matter
    clover = gdm - green            # dry clover = green dry matter - dry green biomass
    
    final_preds_5.append([green, dead, clover, gdm, total])

# Map to Submission
pred_map = {path: preds for path, preds in zip(unique_test_df['image_path'], final_preds_5)}
target_indices = {'Dry_Green_g': 0, 'Dry_Dead_g': 1, 'Dry_Clover_g': 2, 'GDM_g': 3, 'Dry_Total_g': 4}

submission_rows = []
for _, row in raw_test_df.iterrows():
    img_name = row['image_path']
    t_name = row['target_name']
    val = pred_map[img_name][target_indices[t_name]]
    submission_rows.append({'sample_id': row['sample_id'], 'target': float(val)})

sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv('submission.csv', index=False)
print(sub_df.head())

                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.000000
1    ID1001187975__Dry_Dead_g  21.953606
2   ID1001187975__Dry_Green_g  33.913876
3   ID1001187975__Dry_Total_g  55.867481
4         ID1001187975__GDM_g  33.913876
